In [ ]:
# Example 1: Top-level public API surface — verify exports exist.
import ollama

expected = [
    'Client', 'AsyncClient',
    'chat', 'generate', 'embed', 'embeddings',
    'list', 'show', 'ps', 'pull', 'push', 'create', 'copy', 'delete',
    'Message', 'Options', 'Tool',
    'ResponseError', 'RequestError',
]
missing = [name for name in expected if not hasattr(ollama, name)]
present = [name for name in expected if hasattr(ollama, name)]
print(f"Present: {len(present)}/{len(expected)}")
for n in present:
    print(f"  {n}: {type(getattr(ollama, n)).__name__}")
if missing:
    print(f"Missing: {missing}")
# Core surfaces must exist
assert hasattr(ollama, 'Client')
assert hasattr(ollama, 'AsyncClient')
assert hasattr(ollama, 'ResponseError')


In [ ]:
# Example 2: Client / AsyncClient instantiation (no network call).
from ollama import Client, AsyncClient

client = Client(host='http://localhost:11434')
print(f"Client type: {type(client).__name__}")
print(f"Client has chat: {hasattr(client, 'chat')}")
print(f"Client has generate: {hasattr(client, 'generate')}")
print(f"Client has embed: {hasattr(client, 'embed')}")
print(f"Client has list: {hasattr(client, 'list')}")

aclient = AsyncClient(host='http://localhost:11434')
print(f"\nAsyncClient type: {type(aclient).__name__}")
print(f"AsyncClient has chat: {hasattr(aclient, 'chat')}")
print(f"AsyncClient has generate: {hasattr(aclient, 'generate')}")

assert isinstance(client, Client)
assert isinstance(aclient, AsyncClient)


In [ ]:
# Example 3: Pydantic-typed Message / Options models.
from ollama import Message, Options

msg = Message(role='user', content='Hello, world!')
print(f"Message: {msg}")
print(f"  role={msg.role!r}")
print(f"  content={msg.content!r}")

msg_dump = msg.model_dump()
print(f"\nmodel_dump: {msg_dump}")
assert msg_dump['role'] == 'user'
assert msg_dump['content'] == 'Hello, world!'

opts = Options(temperature=0.7, num_predict=128, top_p=0.9)
print(f"\nOptions: temperature={opts.temperature}, num_predict={opts.num_predict}")
assert opts.temperature == 0.7
assert opts.num_predict == 128


In [ ]:
# Example 4: Exception classes — verify hierarchy + construction.
from ollama import ResponseError, RequestError

# ResponseError is raised on non-2xx; takes (error, status_code)
err = ResponseError('model not found', 404)
print(f"ResponseError: {err}")
print(f"  status_code: {err.status_code}")
print(f"  error: {err.error}")
assert err.status_code == 404
assert isinstance(err, Exception)

# RequestError is for client-side argument problems
rerr = RequestError('invalid arg')
print(f"\nRequestError: {rerr}")
assert isinstance(rerr, Exception)

# Both should be catchable as plain Exception
try:
    raise ResponseError('boom', 500)
except Exception as e:
    print(f"\nCaught: {type(e).__name__}: {e}")
    assert isinstance(e, ResponseError)


In [ ]:
# Example 5: Network error path — chat() against unreachable host raises cleanly.
import httpx

from ollama import Client, ResponseError

# Port 1 is reserved; connection will fail fast.
client = Client(host='http://127.0.0.1:1', timeout=2.0)

# Only connection-level errors are expected; anything else (TypeError, AttributeError, ...) fails the cell.
try:
    client.chat(model='llama3', messages=[{'role': 'user', 'content': 'hi'}])
except (ResponseError, httpx.ConnectError, httpx.TimeoutException, ConnectionError) as e:
    print(f"Expected error: {type(e).__name__}: {str(e)[:120]}")
else:
    raise AssertionError("client.chat against an unreachable host must raise")

print("\nOK: ollama client surfaces connection errors as expected")
